# Level-2 数据质量复核
## 摘要
本笔记本复用网页的同一计算器，检查目标日三表覆盖、未知方向、时钟、候选关联键、三表全字段重复／Parquet物理行时钟回退；不是数据完整性认证。执行后的结果以本地来源指纹为准。

## 背景与方法
只读扫描沪深A股；按股票、目标日期识别数据。未知方向不填零，无效关联编号不拼成大单。全部SQL保存在同目录 `level2_quality.py`，共同有效成交规则在 `level2_contract.py`。
### 关键假设
原生结构不含频道，成交编号仅为候选唯一键；原始字段完全重复仍不能自动删除，委托编号重复更可能来自不同事件。三表按股票批量检查，文件物理顺序不等于交易所事件顺序。旧快照不覆盖。首次执行会扫描大文件，之后仅复用来源与代码指纹一致的缓存。

## 数据与参数
在仓库目录执行；实际路径由本地 `level2_paths.json` 提供。非股票标的排除。

In [1]:
import duckdb, pandas as pd
from level2_paths import PATHS
from level2_quality import quality_report
target_day = '20260922'
connection = duckdb.connect()
connection.execute("SET memory_limit='4GB'")
connection.execute('SET threads=2')
connection.execute('SET preserve_insertion_order=false')
try:
    quality = quality_report(connection, PATHS['level2'], target_day)
finally:
    connection.close()
print('来源指纹:', quality['sourceIdentity'])

来源指纹: 5a7383af0f6169f6b15f0b4c680bd545f97b6214c2fe5db928ddf06836bc6db2


## 结果
分母是每张表的A股行数。盘后行、零时钟不直接判错；各表缺证券相对于三表并集，并非相对于所有应上市股票。

In [2]:
display(pd.DataFrame(quality['tables']).T[['stocks','rows','badDate','invalidClock','zeroClock','afterClose']])
display(pd.DataFrame([quality['direction']]))
display(pd.DataFrame([quality['candidateDuplicateKey']]))
display(pd.DataFrame([quality['snapshotTimestampDuplicates']]))
display(pd.DataFrame([quality['snapshotExactDuplicates']]))
display(pd.DataFrame([quality['snapshotPhysicalTimeRegressions']]))
display(pd.DataFrame({kind: quality.get(kind + 'ExactDuplicates') for kind in ['deal', 'orderRaw']}).T)
display(pd.DataFrame({kind: quality.get(kind + 'PhysicalTimeRegressions') for kind in ['deal', 'orderRaw']}).T)
display(pd.DataFrame(quality['linkage']).T)
print('共同覆盖:', quality['coverage']['intersection'], '并集:', quality['coverage']['union'])
print('缺少证券:', quality['coverage']['missingByTable'])

,stocks,rows,badDate,invalidClock,zeroClock,afterClose
deal,5209,175765103,0,0,0,35465
snapshot,5209,20876485,0,0,0,101452
order_raw,5209,222969129,0,0,2313,2313


,validTrades,amount,knownNet,unknownAmount,unknownRate,invalidLinkedKeyTrades
0,144148354,2.135908e+12,-7.258816e+10,0.0,0.0,0


,groups,affectedRows,excessRows,rate
0,14,28,14,1.593035e-07


,eligibleRows,groups,affectedStocks,affectedRows,excessRows
0,20403765,0,0,0,0


,eligibleRows,candidateKeyGroups,candidateRows,groups,affectedRows,excessRows
0,20876485,53,106,0,0,0


,eligibleRows,comparablePairs,regressions,affectedStocks
0,20403765,20398556,0,0


,eligibleRows,groups,affectedRows,excessRows,affectedStocks
deal,175765103,0,0,0,0
orderRaw,222969129,0,0,0,0


,eligibleRows,comparablePairs,regressions,affectedStocks
deal,174846393,174841184,0,0
orderRaw,218041320,218036111,0,0


,eligibleTrades,matchedTrades,repeatedKeyTrades,multiSideKeyTrades,matchRate
委托编号,144148354.0,369278.0,0.0,0.0,0.002562
交易所委托号,144148354.0,88263723.0,877146.0,0.0,0.612312


共同覆盖: 5209 并集: 5209
缺少证券: {'deal': [], 'snapshot': [], 'order_raw': []}


## 如何使用
方向未知影响净额、候选标签与连续状态，不能推断为平衡；缺行情影响覆盖，不能推断为停牌。候选键重复必须结合频道/事件语义继续核验，不通过删行使结果看起来干净。
同股同时间快照重复仅指时间键相同，不证明原始事件相同；分母限定目标日连续竞价。三表全字段重复只判定字节语义上的原始字段相同，不自动去重；物理行时钟回退只描述Parquet文件排列，不等于交易所源事件顺序。三表源事件顺序、委托类型及盘口重建尚未完成；不升级为补撤单或吸筹确认。